# 07_mlprep_v4

Robust, configurable ML preprocessing notebook for **Logistic Regression** and **XGBoost** using `ColumnTransformer`.

Key properties:

- no hardcoded target assumptions; you provide `label_cols`
- auditable include/exclude policy with a review report
- separate feature blocks: `time`, `numeric`, `binary`, `categorical`
- separate preprocessors for LR and XGB
- pre/post transformation QA summaries

In [45]:
# Core imports
from dataclasses import dataclass, field
from typing import List, Dict, Any, Pattern, Optional, Tuple
from pathlib import Path

import re
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from icare_risk.utils.io import load_pkg_yaml

In [46]:
# Load dataset
schema = load_pkg_yaml("config/icare/schema.yaml")
DATA_DIR = Path(schema["data_root"])

df = pd.read_parquet(DATA_DIR / "03_derived_analysis_dataset.parquet").copy()
print(f"Loaded dataset shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
print(df.columns.tolist())

Loaded dataset shape: (10000, 186)
Columns: 186
['SUBJECT', 'SPELL_IDENTIFIER', 'ENCNTR_ID', 'EPISODE_IDENTIFIER', 'ADMISSION_DATE', 'ADMISSION_TIME', 'AGE_AT_ADMISSION', 'DISCHARGE_DATE', 'MAIN_SPECIALTY_CODE', 'MAIN_SPECIALTY_CODE_DESC', 'INDEX_OF_MULTIPLE_DEPRIVATION_DECILE', 'index_admission', 'definitions', 'charlson_hx_mi', 'charlson_hx_chf', 'charlson_hx_pvd', 'charlson_hx_stroke', 'charlson_hx_dementia', 'charlson_hx_pulmonary', 'charlson_hx_rheum', 'charlson_hx_pud', 'charlson_hx_liver_mild', 'charlson_hx_diabetes_uncomp', 'charlson_hx_hemiplegia', 'charlson_hx_renal_mod_sev', 'charlson_hx_diabetes_comp', 'charlson_hx_cancer_solid', 'charlson_hx_leukemia', 'charlson_hx_lymphoma', 'charlson_hx_liver_mod_sev', 'charlson_hx_cancer_met', 'charlson_hx_aids', 'charlson_hx_hiv', 'kim_hx_prior_esbl', 'kim_hx_recent_hosp_1yr', 'kim_hx_nursing_home_resident', 'kim_hx_urinary_catheter', 'kim_hx_prior_antibiotics_90d', 'kim_hx_recent_antibiotics_90d', 'tumbarello_hx_prior_esbl', 'tumbarel

In [47]:
# ============================================================
# 2) Config (single control point)
# ============================================================
@dataclass
class MLFeatureConfig:
    # Labels that you choose manually (one or many)
    label_cols: List[str] = field(default_factory=lambda: ["esbl_label_test"])

    # Optional mapping to binary for specific label columns:
    # Example: {"esbl_label_test": "ESBL"}
    label_positive_map: Dict[str, Any] = field(default_factory=dict)

    # Hard exclusions
    id_cols: List[str] = field(default_factory=lambda: [
        "SUBJECT", "SPELL_IDENTIFIER", "ENCNTR_ID", "EPISODE_IDENTIFIER"
    ])
    datetime_cols: List[str] = field(default_factory=lambda: [
        "ADMISSION_DATE", "ADMISSION_TIME", "DISCHARGE_DATE",
        "BIRTH_DATE", "DEATH_DATE", "UPDATE_DT_TM",
        "closest_culture_date", "index_admission"
    ])

    # Optional explicit categoricals
    explicit_categoricals: List[str] = field(default_factory=lambda: [
        "MAIN_SPECIALTY_CODE", "MAIN_SPECIALTY_CODE_DESC",
        "INDEX_OF_MULTIPLE_DEPRIVATION_DECILE",
        "ETHNICITY", "ETHNICITY_DES", "GENDER", "GENDER_DESC",
        "definitions", "qualifying_last_abx"
    ])

    # Optional redundant pairs to drop
    drop_redundant_cols: List[str] = field(default_factory=lambda: [
        "MAIN_SPECIALTY_CODE", "ETHNICITY", "GENDER"
    ])

    # Time-like patterns
    time_patterns: Tuple[str, ...] = ("_dslast", "_dsfirst", "days_since", "time_to_culture")

    # Governance controls
    manual_exclude_cols: List[str] = field(default_factory=list)
    holdout_review_cols: List[str] = field(default_factory=list)
    drop_regex: List[Pattern] = field(default_factory=lambda: [
        re.compile(r".*target.*", re.IGNORECASE)
    ])

    # Quality filters
    max_missing_pct: float = 95.0
    min_unique_non_null: int = 2


# Create the ML configuration
cfg = MLFeatureConfig(
    label_cols=["esbl_label_test"],
    label_positive_map={},
    manual_exclude_cols=[
        "definitions", "time_to_culture_days",
        "time_to_culture_hours", "esbl_status"
    ],
    holdout_review_cols=[]
)

print("Config ready.")

Config ready.


In [48]:
# ============================================================
# 3) Feature planning (include/exclude + grouped blocks)
# ============================================================
def build_ml_feature_plan(df: pd.DataFrame, cfg: MLFeatureConfig):
    df_work = df.copy()

    # Validate labels
    missing_labels = [c for c in cfg.label_cols if c not in df_work.columns]
    if missing_labels:
        raise ValueError(f"Missing label columns in df: {missing_labels}")

    # Build y (single or multi-label)
    y = df_work[cfg.label_cols].copy()
    for col, pos_val in cfg.label_positive_map.items():
        if col not in y.columns:
            raise ValueError(f"label_positive_map key '{col}' is not in label_cols")
        y[col] = (df_work[col] == pos_val).astype(int)

    all_cols = list(df_work.columns)
    reasons = {c: [] for c in all_cols}

    base_exclusions = set(
        cfg.id_cols
        + cfg.datetime_cols
        + cfg.label_cols
        + cfg.drop_redundant_cols
        + cfg.manual_exclude_cols
        + cfg.holdout_review_cols
    )

    # reason flags
    for c in all_cols:
        if c in cfg.id_cols:
            reasons[c].append("id_col")
        if c in cfg.datetime_cols:
            reasons[c].append("datetime_col")
        if c in cfg.label_cols:
            reasons[c].append("label_col")
        if c in cfg.drop_redundant_cols:
            reasons[c].append("redundant_col")
        if c in cfg.manual_exclude_cols:
            reasons[c].append("manual_exclude")
        if c in cfg.holdout_review_cols:
            reasons[c].append("holdout_review")
        for rx in cfg.drop_regex:
            if rx.match(c):
                reasons[c].append(f"drop_regex:{rx.pattern}")

    candidate_cols = [c for c in all_cols if c not in base_exclusions]

    # quality gates
    for c in candidate_cols:
        miss_pct = 100.0 * df_work[c].isna().mean()
        nunique_non_null = int(df_work[c].dropna().nunique())
        if miss_pct > cfg.max_missing_pct:
            reasons[c].append(f"high_missing>{cfg.max_missing_pct}%")
        if nunique_non_null < cfg.min_unique_non_null:
            reasons[c].append("constant_or_single_value")

    kept = [c for c in candidate_cols if len(reasons[c]) == 0]

    # grouped blocks
    time_cols = [c for c in kept if any(p in c for p in cfg.time_patterns)]

    explicit_cat = [c for c in cfg.explicit_categoricals if c in kept]
    object_cat = [c for c in kept if str(df_work[c].dtype) in ("object", "category")]
    categorical_initial = sorted(set(explicit_cat + object_cat))

    binary_cols = []
    for c in kept:
        if c in time_cols or c in categorical_initial:
            continue
        s = df_work[c]
        if pd.api.types.is_numeric_dtype(s):
            vals = pd.Series(s.dropna().unique())
            if len(vals) > 0 and len(vals) <= 2 and set(vals.astype(float)).issubset({0.0, 1.0}):
                binary_cols.append(c)

    numeric_cols = [
        c for c in kept
        if c not in time_cols
        and c not in categorical_initial
        and c not in binary_cols
        and pd.api.types.is_numeric_dtype(df_work[c])
    ]

    categorical_cols = [c for c in categorical_initial]

    feature_cols = time_cols + numeric_cols + binary_cols + categorical_cols
    feature_cols = list(dict.fromkeys(feature_cols))

    X = df_work[feature_cols].copy()

    # report
    block = {}
    for c in time_cols:
        block[c] = "time"
    for c in numeric_cols:
        block[c] = "numeric"
    for c in binary_cols:
        block[c] = "binary"
    for c in categorical_cols:
        block[c] = "categorical"

    feature_review = pd.DataFrame({
        "column": all_cols,
        "dtype": [str(df_work[c].dtype) for c in all_cols],
        "included_in_ml": [c in feature_cols for c in all_cols],
        "block": [block.get(c, "excluded") for c in all_cols],
        "reasons_excluded": [";".join(reasons[c]) if reasons[c] else "" for c in all_cols],
        "missing_pct": [round(100.0 * df_work[c].isna().mean(), 3) for c in all_cols],
        "nunique_non_null": [int(df_work[c].dropna().nunique()) for c in all_cols],
    }).sort_values(["included_in_ml", "block", "column"], ascending=[False, True, True])

    groups = {
        "time_cols": time_cols,
        "numeric_cols": numeric_cols,
        "binary_cols": binary_cols,
        "categorical_cols": categorical_cols,
        "feature_cols": feature_cols,
        "label_cols": cfg.label_cols,
    }

    return X, y, groups, feature_review


X, y, groups, feature_review = build_ml_feature_plan(df, cfg)

print("Feature grouping summary")
print("------------------------")
print(f"Time columns       : {len(groups['time_cols'])}")
print(f"Numeric columns    : {len(groups['numeric_cols'])}")
print(f"Binary flag columns: {len(groups['binary_cols'])}")
print(f"Categorical columns: {len(groups['categorical_cols'])}")
print(f"Total features (X) : {X.shape[1]}")
print(f"Labels (y) columns : {list(y.columns)}")

display(feature_review.head(60))

Feature grouping summary
------------------------
Time columns       : 12
Numeric columns    : 15
Binary flag columns: 49
Categorical columns: 4
Total features (X) : 80
Labels (y) columns : ['esbl_label_test']


,column,dtype,included_in_ml,block,reasons_excluded,missing_pct,nunique_non_null
31,charlson_hx_aids,int64,True,binary,,0.0,2
137,charlson_hx_aids_cnt,int64,True,binary,,0.0,2
30,charlson_hx_cancer_met,int64,True,binary,,0.0,2
134,charlson_hx_cancer_met_cnt,int64,True,binary,,0.0,2
26,charlson_hx_cancer_solid,int64,True,binary,,0.0,2
122,charlson_hx_cancer_solid_cnt,int64,True,binary,,0.0,2
14,charlson_hx_chf,int64,True,binary,,0.0,2
17,charlson_hx_dementia,int64,True,binary,,0.0,2
98,charlson_hx_dementia_cnt,int64,True,binary,,0.0,2
25,charlson_hx_diabetes_comp,int64,True,binary,,0.0,2


In [49]:
# ============================================================
# 4) Preprocessors (LR vs XGB)
# ============================================================
def build_preprocessors(groups):
    time_cols = groups["time_cols"]
    numeric_cols = groups["numeric_cols"]
    binary_cols = groups["binary_cols"]
    categorical_cols = groups["categorical_cols"]

    # Keep OHE sparse for memory efficiency
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)

    lr_preprocessor = ColumnTransformer(
        transformers=[
            ("time", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=-1.0)),
                ("scaler", StandardScaler())
            ]), time_cols),
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), numeric_cols),
            ("bin", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent"))
            ]), binary_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
                ("onehot", ohe)
            ]), categorical_cols),
        ],
        remainder="drop"
    )

    xgb_preprocessor = ColumnTransformer(
        transformers=[
            ("time", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=-1.0))
            ]), time_cols),
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]), numeric_cols),
            ("bin", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent"))
            ]), binary_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
                ("onehot", ohe)
            ]), categorical_cols),
        ],
        remainder="drop"
    )

    return lr_preprocessor, xgb_preprocessor

lr_preprocessor, xgb_preprocessor = build_preprocessors(groups)
print("Preprocessors ready.")

Preprocessors ready.


In [50]:
# ============================================================
# 5) Inspection utilities
# ============================================================
def inspect_preprocessor_output(preprocessor,
                                X_input: pd.DataFrame,
                                model_name: str,
                                sample_rows: int = 5):
    X_t = preprocessor.fit_transform(X_input)

    # names
    feat_names = preprocessor.get_feature_names_out()
    feat_names = [n.replace("time__", "time__")
                    .replace("num__", "num__")
                    .replace("bin__", "bin__")
                    .replace("cat__", "cat__")
                  for n in feat_names]

    # sparse/dense diagnostics
    is_sparse = sparse.issparse(X_t)
    if is_sparse:
        n_rows, n_cols = X_t.shape
        density = X_t.nnz / (n_rows * n_cols)
        nan_count = int(np.isnan(X_t.data).sum())
        inf_count = int(np.isinf(X_t.data).sum())
    else:
        density = 1.0
        nan_count = int(np.isnan(X_t).sum())
        inf_count = int(np.isinf(X_t).sum())

    print(f"\n=== {model_name} ===")
    print(f"Input shape     : {X_input.shape}")
    print(f"Output shape    : {X_t.shape}")
    print(f"Sparse output   : {is_sparse}")
    print(f"Density         : {density:.6f}")
    print(f"NaN count       : {nan_count}")
    print(f"Inf count       : {inf_count}")

    # Small preview as DataFrame
    if is_sparse:
        preview = pd.DataFrame(X_t[:sample_rows].toarray(), columns=feat_names)
    else:
        preview = pd.DataFrame(X_t[:sample_rows], columns=feat_names)

    return X_t, preview, feat_names


def build_feature_profile(X: pd.DataFrame, quantiles=(0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99)):
    rows = []
    cat_level_rows = []

    q_cols = [f"q{int(q*100):02d}" for q in quantiles]

    for col in X.columns:
        s = X[col]
        n = len(s)
        missing_n = s.isna().sum()
        missing_pct = (missing_n / n) * 100

        non_null = s.dropna()
        as_str = non_null.astype(str).str.lower().str.strip()
        binary_tokens = {"0", "1", "true", "false"}
        is_binary = len(non_null) > 0 and set(as_str.unique()).issubset(binary_tokens)
        is_numeric = pd.api.types.is_numeric_dtype(s)

        base = {
            "feature": col,
            "count": n,
            "missing_n": missing_n,
            "missing_pct": round(missing_pct, 3),
            "n_unique_non_missing": non_null.nunique()
        }

        if is_binary:
            mapped = as_str.map({"1": 1, "true": 1, "0": 0, "false": 0})
            row = {
                **base,
                "type": "binary",
                "positive_pct": round((mapped == 1).mean() * 100, 3) if len(mapped) else np.nan,
                "negative_pct": round((mapped == 0).mean() * 100, 3) if len(mapped) else np.nan,
                "mean": np.nan, "std": np.nan, "min": np.nan, "median": np.nan, "max": np.nan,
            }
            row.update({qc: np.nan for qc in q_cols})
            rows.append(row)

        elif is_numeric:
            nn = s.dropna()
            qvals = nn.quantile(list(quantiles)) if len(nn) else pd.Series(index=quantiles, dtype=float)
            row = {
                **base,
                "type": "numeric",
                "positive_pct": np.nan, "negative_pct": np.nan,
                "mean": round(nn.mean(), 6) if len(nn) else np.nan,
                "std": round(nn.std(ddof=1), 6) if len(nn) > 1 else np.nan,
                "min": round(nn.min(), 6) if len(nn) else np.nan,
                "median": round(nn.median(), 6) if len(nn) else np.nan,
                "max": round(nn.max(), 6) if len(nn) else np.nan,
            }
            row.update({f"q{int(q*100):02d}": round(qvals.loc[q], 6) if len(nn) else np.nan for q in quantiles})
            rows.append(row)

        else:
            row = {
                **base,
                "type": "categorical",
                "positive_pct": np.nan, "negative_pct": np.nan,
                "mean": np.nan, "std": np.nan, "min": np.nan, "median": np.nan, "max": np.nan,
            }
            row.update({qc: np.nan for qc in q_cols})
            rows.append(row)

            s_cat = s.astype("object").where(~s.isna(), "Missing")
            vc = s_cat.value_counts(dropna=False)
            for lvl, cnt in vc.items():
                cat_level_rows.append({
                    "feature": col,
                    "level": lvl,
                    "count": int(cnt),
                    "pct": round((cnt / n) * 100, 3)
                })

    feature_summary = pd.DataFrame(rows).sort_values(["type", "feature"]).reset_index(drop=True)
    categorical_levels = pd.DataFrame(cat_level_rows).sort_values(
        ["feature", "count"], ascending=[True, False]
    ).reset_index(drop=True)

    return feature_summary, categorical_levels

In [51]:
# ============================================================
# 6) RUN & TEST BOTH PREPROCESSORS
# ============================================================
# Test Linear Regression pipeline
X_lr_t, lr_preview, lr_feature_names = inspect_preprocessor_output(
    lr_preprocessor, X, sample_rows=5,
    model_name="Logistic Regression preprocessing"
)

# Text XGBoost pipeline
X_xgb_t, xgb_preview, xgb_feature_names = inspect_preprocessor_output(
    xgb_preprocessor, X, sample_rows=5,
    model_name="XGBoost preprocessing",
)


=== Logistic Regression preprocessing ===
Input shape     : (10000, 80)
Output shape    : (10000, 108)
Sparse output   : False
Density         : 1.000000
NaN count       : 0
Inf count       : 0

=== XGBoost preprocessing ===
Input shape     : (10000, 80)
Output shape    : (10000, 108)
Sparse output   : False
Density         : 1.000000
NaN count       : 0
Inf count       : 0


In [52]:
# Pandas show all configuration
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

# Raw feature profile (before transformer)
feature_summary_df, categorical_levels_df = build_feature_profile(X)

print("\nRaw feature summary:")
display(feature_summary_df)

print("\nCategorical levels (first 100):")
display(categorical_levels_df.head(100))


Raw feature summary:


,feature,count,missing_n,missing_pct,n_unique_non_missing,type,positive_pct,negative_pct,mean,std,min,median,max,q01,q05,q10,q25,q50,q75,q90,q95,q99
0,charlson_hx_aids,10000,0,0.00,2,binary,0.15,99.85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,charlson_hx_aids_cnt,10000,0,0.00,2,binary,0.15,99.85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,charlson_hx_cancer_met,10000,0,0.00,2,binary,0.04,99.96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,charlson_hx_cancer_met_cnt,10000,0,0.00,2,binary,0.04,99.96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,charlson_hx_cancer_solid,10000,0,0.00,2,binary,0.04,99.96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,charlson_hx_cancer_solid_cnt,10000,0,0.00,2,binary,0.04,99.96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,charlson_hx_chf,10000,0,0.00,2,binary,0.55,99.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,charlson_hx_dementia,10000,0,0.00,2,binary,0.77,99.23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,charlson_hx_dementia_cnt,10000,0,0.00,2,binary,0.77,99.23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,charlson_hx_diabetes_comp,10000,0,0.00,2,binary,0.08,99.92,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Categorical levels (first 100):


,feature,level,count,pct
0,ETHNICITY_DES,White - British,872,8.72
1,ETHNICITY_DES,Any other Black background,867,8.67
2,ETHNICITY_DES,Black or Black British - Caribbean,854,8.54
3,ETHNICITY_DES,Not stated,848,8.48
4,ETHNICITY_DES,White - Irish,846,8.46
5,ETHNICITY_DES,Asian or Asian British - Bangladeshi,841,8.41
6,ETHNICITY_DES,Any other White background,822,8.22
7,ETHNICITY_DES,Any other ethnic group,814,8.14
8,ETHNICITY_DES,Asian or Asian British - Indian,813,8.13
9,ETHNICITY_DES,Black or Black British - African,809,8.09


In [53]:
# ============================================================
# 7) Save QA artifacts
# ============================================================
# Suggested folder name
output_dir = Path("mlprep_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Save reports
feature_review.to_csv(output_dir / "feature_review_report.csv", index=False)
feature_summary_df.to_csv(output_dir / "feature_summary_full.csv", index=False)
categorical_levels_df.to_csv(output_dir / "categorical_levels_full.csv", index=False)

# Save transformed previews only (full transformed matrix can be large)
lr_preview.to_csv(output_dir / "lr_transformed_preview.csv", index=False)
xgb_preview.to_csv(output_dir / "xgb_transformed_preview.csv", index=False)

print("Saved files in:", output_dir.resolve())
print("- feature_review_report.csv")
print("- feature_summary_full.csv")
print("- categorical_levels_full.csv")
print("- lr_transformed_preview.csv")
print("- xgb_transformed_preview.csv")

Saved files in: /app/notebooks/b_preparation/mlprep_outputs
- feature_review_report.csv
- feature_summary_full.csv
- categorical_levels_full.csv
- lr_transformed_preview.csv
- xgb_transformed_preview.csv


In [54]:
import json
from pathlib import Path
from datetime import datetime, timezone

manifest = {
    "version": 1,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "label_cols": cfg.label_cols,  # from your MLFeatureConfig
    "feature_cols": groups["feature_cols"],
    "groups": {
        "time_cols": groups["time_cols"],
        "numeric_cols": groups["numeric_cols"],
        "binary_cols": groups["binary_cols"],
        "categorical_cols": groups["categorical_cols"],
    },
    "governance": {
        "manual_exclude_cols": cfg.manual_exclude_cols,
        "holdout_review_cols": cfg.holdout_review_cols,
        "drop_redundant_cols": cfg.drop_redundant_cols,
        "max_missing_pct": cfg.max_missing_pct,
        "min_unique_non_null": cfg.min_unique_non_null,
        "drop_regex": [r.pattern for r in cfg.drop_regex],
    },
}

manifest_path = output_dir / "feature_manifest.json"
with manifest_path.open("w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(f"Saved manifest: {manifest_path.resolve()}")

Saved manifest: /app/notebooks/b_preparation/mlprep_outputs/feature_manifest.json
